# controlcharts walkthrough

A short tour of the package. Everything here imports from `controlcharts`;
no logic lives in this notebook.

Install the package first:

```bash
pip install -e ".[dev]"
```

In [ ]:
from controlcharts.data import CLASS_LABELS, load_csv
from controlcharts.evaluate import evaluate, format_report
from controlcharts.model import load_model
from controlcharts.plots import plot_confusion_matrix, plot_control_charts, plot_example_patterns
from controlcharts.predict import analyze
from controlcharts.preprocess import prepare

print(CLASS_LABELS)

## 1. The dataset

600 series of 60 points, 100 per class.

In [ ]:
dataset = load_csv("../data/synthetic_control.csv")
print(dataset.series.shape, len(set(dataset.labels)), "classes")

In [ ]:
plot_example_patterns(dataset, "../assets/example_patterns.png")

![patterns](../assets/example_patterns.png)

## 2. Score the trained model

Each series is divided by its own peak, so only its shape matters. The split
is stratified and seeded, which the original notebook's was not.

In [ ]:
split = prepare(dataset, test_size=0.10, seed=101)
model = load_model("../models/control_charts.keras")
evaluation = evaluate(model, split.x_test, split.y_test)
print(format_report(evaluation))

In [ ]:
plot_confusion_matrix(evaluation, "../assets/confusion_matrix.png")

![confusion](../assets/confusion_matrix.png)

## 3. Score a real process

The diameter measurements come off a production line with a 39.9 to 40.1 mm
tolerance. `analyze` builds the X-bar and R charts, computes capability, and
classifies both chart series.

In [ ]:
report = analyze(model, "../data/diameter_real_process.csv", usl=40.1, lsl=39.9)

print(f"grand mean {report.chart.center_x:.4f} mm")
print(f"mean range {report.chart.center_r:.4f} mm")
print(f"Cp {report.capability.cp:.3f}, Cpk {report.capability.cpk:.3f}")

for prediction in report.predictions:
    print(f"{prediction.chart:>6}: {prediction.pattern} ({prediction.confidence:.1%})")

print(f"in control: {report.in_control}, {len(report.violations)} finding(s)")
for line in report.violations[:5]:
    print(" -", line)

In [ ]:
plot_control_charts(report, "../assets/diameter_control_chart.png", name="diameter")

![charts](../assets/diameter_control_chart.png)